# 

## Imports

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import requests
import time
import os

## Extracting Weather Data with Open\-Meteo

- The relevant weather variables we'll extract are the ones shown below. Again, like before, I'm open to adding more!
    - cloud_cover_pct
    - wind_speed_kmh
    - rel_humidity_pct
    - precipitation_mm

## Using the API

One thing to keep in mind is that on the free tier, we are limited to 10,000 API calls a day, so try to make these API calls sparingly\!

In [3]:
# KEY : VALUE
# Open-Meteo API variable name : DataFrame column name
extracted_variables = {
    "cloud_cover": "cloud_cover_pct",
    "wind_speed_10m": "wind_speed_kmh",
    "relative_humidity_2m": "rel_humidity_pct",
    "precipitation": "precipitation_mm",
}

# Prepare - Add empty columns for weather variables
weather_df = pd.read_csv("data/Processed/aurora_clean.csv")
for key, value in extracted_variables.items():
    weather_df[value] = None
weather_df.head()

,timestamp,lat,lon,mag_lat,mag_lon,mag_lt,aurora_observed,cloud_cover_pct,wind_speed_kmh,rel_humidity_pct,precipitation_mm
0,2014-10-01 08:21:34.818216+00:00,25.041672,55.154847,19.985358,128.137506,12.129542,False,None,None,None,None
1,2014-10-02 13:47:29.552583+00:00,35.874132,-106.318648,44.155324,-39.100305,6.438991,False,None,None,None,None
2,2014-10-08 18:09:23.114761+00:00,35.977794,-106.087598,44.298471,-38.831876,10.782292,True,None,None,None,None
3,2014-10-08 18:11:38.395952+00:00,35.858267,-106.335363,44.136071,-39.115849,10.763361,True,None,None,None,None
4,2014-10-08 18:15:51.473713+00:00,35.869407,-106.336213,44.147181,-39.118361,10.763193,True,None,None,None,None


In [7]:
weather_vars = list(extracted_variables.keys())
weather_vars_str = ",".join(weather_vars)
url = "https://archive-api.open-meteo.com/v1/archive"

def extract_weather_data(row):
    dt = pd.to_datetime(row["timestamp"])
    date_str = dt.strftime("%Y-%m-%d")
    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": date_str,
        "end_date": date_str,
        "hourly": weather_vars_str
    }
    response = requests.get(url, params=params)
    data = response.json()
    hour = dt.hour
    return [data["hourly"][var][hour] for var in weather_vars]

In [9]:
# QUICK NOTE: We changed around these parameters manually on a schedule to make sure we don't exceed the API limit
# The code in this block yields eight batches, which is around 4000 entries.

# We are rate-limited by 600 requests a minute, 5000 per hour, 10000 a day.
# We have ~20,000 observations to work with...
minute_limit = 500

# Change the offset to where we last left off, whenever this block is run. Refer to weather_batches
offset = 0
hour_iters = 8 # Should put as at around 4000 processed, we can run this every hour

for i in range(hour_iters):
    if offset >= len(weather_df):
        break
    
    print("Batch hour iteration", i)
    end = min(offset + minute_limit, len(weather_df))
    batch = weather_df.iloc[offset:end].copy()
    print(f"Processing rows {offset}–{end-1}...")

    filename = f"batch_{offset}-{end-1}.csv"

    # Fetch data
    batch[[extracted_variables[key] for key in weather_vars]] = pd.DataFrame(
        batch.apply(extract_weather_data, axis=1).tolist(),
        index=batch.index
    )

    batch.to_csv(f"data/Processed/weather_batches/{filename}", index=False, lineterminator='\n')
    print("Finished API data extraction for", filename, ", writing to file")
    offset = offset + minute_limit
    if i < hour_iters - 1 and offset < len(weather_df):
        print("Waiting two minutes...")
        time.sleep(120)

print("Finished batch processing!")

Batch hour iteration 0
Processing rows 0–499...
Finished API data extraction for batch_0-499.csv , writing to file
Waiting two minutes...
Batch hour iteration 1
Processing rows 500–999...
Finished API data extraction for batch_500-999.csv , writing to file
Waiting two minutes...
Batch hour iteration 2
Processing rows 1000–1499...
Finished API data extraction for batch_1000-1499.csv , writing to file
Waiting two minutes...
Batch hour iteration 3
Processing rows 1500–1999...
Finished API data extraction for batch_1500-1999.csv , writing to file
Waiting two minutes...
Batch hour iteration 4
Processing rows 2000–2499...
Finished API data extraction for batch_2000-2499.csv , writing to file
Waiting two minutes...
Batch hour iteration 5
Processing rows 2500–2999...
Finished API data extraction for batch_2500-2999.csv , writing to file
Waiting two minutes...
Batch hour iteration 6
Processing rows 3000–3499...
Finished API data extraction for batch_3000-3499.csv , writing to file
Waiting two m

## Combine Batches

In [16]:
dir_path = "data/Processed/weather_batches"
file_list = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
file_list = sorted(file_list, key=lambda name: int(name.split("_")[1].split("-")[0]))
file_list

['batch_0-499.csv',
 'batch_500-999.csv',
 'batch_1000-1499.csv',
 'batch_1500-1999.csv',
 'batch_2000-2499.csv',
 'batch_2500-2999.csv',
 'batch_3000-3499.csv',
 'batch_3500-3999.csv',
 'batch_4000-4499.csv',
 'batch_4500-4999.csv',
 'batch_5000-5499.csv',
 'batch_5500-5999.csv',
 'batch_6000-6499.csv',
 'batch_6500-6999.csv',
 'batch_7000-7499.csv',
 'batch_7500-7999.csv',
 'batch_8000-8499.csv',
 'batch_8500-8999.csv',
 'batch_9000-9499.csv',
 'batch_9500-9999.csv',
 'batch_10000-10499.csv',
 'batch_10500-10999.csv',
 'batch_11000-11499.csv',
 'batch_11500-11999.csv',
 'batch_12000-12499.csv',
 'batch_12500-12999.csv',
 'batch_13000-13499.csv',
 'batch_13500-13999.csv',
 'batch_14000-14499.csv',
 'batch_14500-14999.csv',
 'batch_15000-15499.csv',
 'batch_15500-15999.csv',
 'batch_16000-16499.csv',
 'batch_16500-16999.csv',
 'batch_17000-17499.csv',
 'batch_17500-17999.csv',
 'batch_18000-18499.csv',
 'batch_18500-18999.csv',
 'batch_19000-19499.csv',
 'batch_19500-19999.csv',
 'batc

In [22]:
combined_df = pd.read_csv(f"{dir_path}/{file_list[0]}")
print("Initial DF:", f"{dir_path}/{file_list[0]}")
for file in file_list[1:]:
    print(f"Reading {file}...")
    df = pd.read_csv(f"{dir_path}/{file}")
    combined_df = pd.concat([combined_df, df], ignore_index=True)

print("Merge complete!")

Initial DF: Processed/weather_batches/batch_0-499.csv
Reading batch_500-999.csv...
Reading batch_1000-1499.csv...
Reading batch_1500-1999.csv...
Reading batch_2000-2499.csv...
Reading batch_2500-2999.csv...
Reading batch_3000-3499.csv...
Reading batch_3500-3999.csv...
Reading batch_4000-4499.csv...
Reading batch_4500-4999.csv...
Reading batch_5000-5499.csv...
Reading batch_5500-5999.csv...
Reading batch_6000-6499.csv...
Reading batch_6500-6999.csv...
Reading batch_7000-7499.csv...
Reading batch_7500-7999.csv...
Reading batch_8000-8499.csv...
Reading batch_8500-8999.csv...
Reading batch_9000-9499.csv...
Reading batch_9500-9999.csv...
Reading batch_10000-10499.csv...
Reading batch_10500-10999.csv...
Reading batch_11000-11499.csv...
Reading batch_11500-11999.csv...
Reading batch_12000-12499.csv...
Reading batch_12500-12999.csv...
Reading batch_13000-13499.csv...
Reading batch_13500-13999.csv...
Reading batch_14000-14499.csv...
Reading batch_14500-14999.csv...
Reading batch_15000-15499.csv

In [26]:
print(f"Length: {len(combined_df)} rows")
print("NaN Info:")
print(combined_df.isna().sum())

Length: 22264 rows
NaN Info:
timestamp           0
lat                 0
lon                 0
mag_lat             0
mag_lon             0
mag_lt              0
aurora_observed     0
cloud_cover_pct     0
wind_speed_kmh      0
rel_humidity_pct    0
precipitation_mm    0
dtype: int64


In [32]:
combined_df.to_csv("data/Processed/aurora_weather_combined.csv", index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>